# Contagion Simulations

For every bank in every quarter (2016 Q1 – 2023 Q4), apply fixed equity shocks
via `simulate_failure` and store the cascade results as three normalised tables.

| Output directory | Table | Grain |
|---|---|---|
| `src/data/sim_runs/` | **Table 1 – runs** | 1 row per simulation |
| `src/data/sim_bank_state/` | **Table 2 – bank_end_state_sparse** | 1 row per impacted / initial / failed bank |
| `src/data/sim_round_summary/` | **Table 3 – round_summary** | 1 row per cascade round (optional) |

In [1]:
import sys
sys.path.insert(0, '..')

import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_data, build_run_row, build_bank_rows, build_round_rows
from src.models import simulate_failure

In [2]:
# Configuration
N_SIMULATIONS   = 500_000   # simulations per quarter
MECHANISM       = "Exposure"
ALPHA           = 1.0
SPREAD_WO_DEF   = True
SHOCK_MIN       = 0.05
SHOCK_MAX       = 0.80

# Set to True to also write Table 3 (round_summary).
TRACK_ROUNDS    = True

PROJECT_ROOT    = Path().resolve().parent

OUT_RUNS        = PROJECT_ROOT / "src" / "data" / "sim_runs"
OUT_BANK_STATE  = PROJECT_ROOT / "src" / "data" / "sim_bank_state"
OUT_ROUND_SUM   = PROJECT_ROOT / "src" / "data" / "sim_round_summary"

for d in (OUT_RUNS, OUT_BANK_STATE, OUT_ROUND_SUM):
    d.mkdir(parents=True, exist_ok=True)

print("Output dirs:")
print(" ", OUT_RUNS)
print(" ", OUT_BANK_STATE)
print(" ", OUT_ROUND_SUM)

Output dirs:
  C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\data\sim_runs
  C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\data\sim_bank_state
  C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\data\sim_round_summary


In [3]:
# Load all 32 quarters
quarters = {}

for year in range(2016, 2024):
    for q in range(1, 5):
        edges, nodes = load_data(year, q)
        quarters[(year, q)] = (edges, nodes)

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


In [4]:

summary_rows = []

for (year, q), (edges, nodes) in quarters.items():

    # per-quarter constants
    equity_initial = nodes.set_index("index")["Equity"].to_dict()
    n_banks        = len(nodes)
    n_edges        = len(edges)
    bank_ids       = nodes["index"].tolist()

    # Seeded RNG per quarter for reproducibility
    rng            = np.random.default_rng(seed=year * 10 + q)
    sampled_banks  = rng.choice(bank_ids, size=N_SIMULATIONS, replace=True)
    shock_fracs    = rng.uniform(SHOCK_MIN, SHOCK_MAX, size=N_SIMULATIONS)

    # Batch timestamp (one value per quarter, not per simulation)
    batch_ts = datetime.now(timezone.utc).isoformat()

    run_rows   = []
    bank_rows  = []
    round_rows = []

    for bank_id, shock_frac in zip(sampled_banks, shock_fracs):
        run_id = str(uuid.uuid4())

        t0 = time.perf_counter()
        result = simulate_failure(
            int(bank_id), edges, nodes,
            mechanism=MECHANISM,
            alpha=ALPHA,
            spread_without_default=SPREAD_WO_DEF,
            initial_loss_mode="fixed",
            initial_loss_frac=float(shock_frac),
            track_rounds=TRACK_ROUNDS,
        )
        runtime_ms = (time.perf_counter() - t0) * 1_000

        run_rows.append(build_run_row(
            run_id, result, equity_initial, n_banks, n_edges,
            initial_bank=int(bank_id),
            mechanism=MECHANISM,
            alpha=ALPHA,
            spread_without_default=SPREAD_WO_DEF,
            initial_loss_mode="fixed",
            year=year, quarter=q,
            runtime_ms=runtime_ms,
            timestamp_utc=batch_ts,
        ))

        bank_rows.extend(build_bank_rows(
            run_id, result, equity_initial, int(bank_id)
        ))

        if TRACK_ROUNDS:
            round_rows.extend(build_round_rows(run_id, result))

    #  save Table 1 
    df_runs = pd.DataFrame(run_rows)
    out_runs = OUT_RUNS / f"sim_runs_{year}Q{q}.parquet"
    df_runs.to_parquet(out_runs, index=False)

    # save Table 2 
    df_banks = pd.DataFrame(bank_rows)
    out_banks = OUT_BANK_STATE / f"sim_bank_state_{year}Q{q}.parquet"
    df_banks.to_parquet(out_banks, index=False)

    # save Table 3
    if TRACK_ROUNDS and round_rows:
        df_rounds = pd.DataFrame(round_rows)
        out_rounds = OUT_ROUND_SUM / f"sim_round_summary_{year}Q{q}.parquet"
        df_rounds.to_parquet(out_rounds, index=False)
        round_info = f" | {len(df_rounds):,} round rows"
    else:
        round_info = ""

    summary_rows.append({
        "year":                   year,
        "quarter":                q,
        "period":                 f"{year}Q{q}",
        "n_simulations":          N_SIMULATIONS,
        "n_bank_rows":            len(df_banks),
        "pct_initial_default":    df_runs["initial_default"].mean(),
        "avg_cascade_failed":     df_runs["num_failed"].mean(),
        "avg_impacted_share":     df_runs["impacted_share"].mean(),
        "avg_sys_eq_depletion":   df_runs["system_equity_depletion"].mean(),
    })

    print(f"[OK] {year}Q{q} | {N_SIMULATIONS:,} runs | {len(df_banks):,} bank rows{round_info}")

df_summary = (
    pd.DataFrame(summary_rows)
      .sort_values(["year", "quarter"])
      .reset_index(drop=True)
)
display(df_summary)

[OK] 2016Q1 | 500,000 runs | 1,891,409 bank rows | 514,377 round rows
[OK] 2016Q2 | 500,000 runs | 1,846,135 bank rows | 511,721 round rows
[OK] 2016Q3 | 500,000 runs | 1,840,843 bank rows | 510,824 round rows
[OK] 2016Q4 | 500,000 runs | 1,821,599 bank rows | 509,902 round rows
[OK] 2017Q1 | 500,000 runs | 1,819,069 bank rows | 510,050 round rows
[OK] 2017Q2 | 500,000 runs | 1,805,382 bank rows | 509,272 round rows
[OK] 2017Q3 | 500,000 runs | 1,799,082 bank rows | 509,455 round rows
[OK] 2017Q4 | 500,000 runs | 1,821,355 bank rows | 509,832 round rows
[OK] 2018Q1 | 500,000 runs | 1,869,923 bank rows | 511,579 round rows
[OK] 2018Q2 | 500,000 runs | 1,885,787 bank rows | 510,741 round rows
[OK] 2018Q3 | 500,000 runs | 1,883,530 bank rows | 509,483 round rows
[OK] 2018Q4 | 500,000 runs | 1,858,036 bank rows | 509,162 round rows
[OK] 2019Q1 | 500,000 runs | 1,839,416 bank rows | 509,725 round rows
[OK] 2019Q2 | 500,000 runs | 1,880,197 bank rows | 508,713 round rows
[OK] 2019Q3 | 500,00

,year,quarter,period,n_simulations,n_bank_rows,pct_initial_default,avg_cascade_failed,avg_impacted_share,avg_sys_eq_depletion
0,2016,1,2016Q1,500000,1891409,0.000806,0.244312,0.000832,739174.781585
1,2016,2,2016Q2,500000,1846135,0.000000,0.140818,0.000812,487820.959902
2,2016,3,2016Q3,500000,1840843,0.000000,0.098808,0.000810,391175.334230
3,2016,4,2016Q4,500000,1821599,0.000000,0.079784,0.000801,354421.971111
4,2017,1,2017Q1,500000,1819069,0.000000,0.083020,0.000800,311330.105747
5,2017,2,2017Q2,500000,1805382,0.000000,0.073396,0.000794,300612.045460
6,2017,3,2017Q3,500000,1799082,0.000000,0.071574,0.000791,306456.460267
7,2017,4,2017Q4,500000,1821355,0.000000,0.077840,0.000801,296240.618529
8,2018,1,2018Q1,500000,1869923,0.000000,0.108102,0.000822,296102.153997
9,2018,2,2018Q2,500000,1885787,0.000000,0.094858,0.000829,304272.320088


## Verification

In [8]:
# Table 1
df_runs_check = pd.read_parquet(OUT_RUNS / "sim_runs_2016Q1.parquet")
print("Table 1")
print(f"  shape : {df_runs_check.shape}")
print(f"  cols  : {list(df_runs_check.columns)}")
display(df_runs_check.head(3))
display(df_runs_check[[
    "shock_fraction", "initial_default", "rounds",
    "num_failed", "num_impacted", "failed_share",
    "system_equity_depletion", "max_bank_loss",
]].describe())

Table 1
  shape : (500000, 26)
  cols  : ['run_id', 'dataset_id', 'year', 'quarter', 'mechanism', 'initial_bank', 'initial_loss_mode', 'shock_fraction', 'alpha', 'spread_without_default', 'random_state', 'initial_default', 'rounds', 'num_failed', 'num_impacted', 'failed_share', 'impacted_share', 'failed_equity_loss', 'system_equity_depletion', 'avg_loss_per_impacted', 'max_bank_loss', 'max_loss_bank_id', 'n_banks', 'n_edges', 'runtime_ms', 'timestamp_utc']


,run_id,dataset_id,year,quarter,mechanism,initial_bank,initial_loss_mode,shock_fraction,alpha,spread_without_default,...,impacted_share,failed_equity_loss,system_equity_depletion,avg_loss_per_impacted,max_bank_loss,max_loss_bank_id,n_banks,n_edges,runtime_ms,timestamp_utc
0,482993bf-4282-4b77-948a-80463413358c,2016Q1,2016,1,Exposure,3419,fixed,0.252178,1.0,True,...,0.00022,0.0,3670.192689,3670.192689,3670.192689,3419.0,4548,11631,1.8989,2026-03-01T15:30:53.569004+00:00
1,9e5154fe-f8bd-48c4-ad67-4764c3f1517a,2016Q1,2016,1,Exposure,3301,fixed,0.275989,1.0,True,...,0.00022,0.0,3090.524683,3090.524683,3090.524683,3301.0,4548,11631,1.5958,2026-03-01T15:30:53.569004+00:00
2,ef7136bc-6463-4c2f-b56e-b3f5399c30fa,2016Q1,2016,1,Exposure,2507,fixed,0.385556,1.0,True,...,0.00022,0.0,5483.760734,5483.760734,5483.760734,2507.0,4548,11631,1.4601,2026-03-01T15:30:53.569004+00:00


,shock_fraction,rounds,num_failed,num_impacted,failed_share,system_equity_depletion,max_bank_loss
count,500000.000000,500000.000000,500000.000000,500000.000000,500000.000000,5.000000e+05,5.000000e+05
mean,0.425327,1.028754,0.244312,3.782012,0.000054,7.391748e+05,3.639249e+05
std,0.216540,0.190901,4.344799,34.894033,0.000955,9.244077e+06,3.417084e+06
min,0.050003,1.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,0.237777,1.000000,0.000000,1.000000,0.000000,3.761829e+03,3.607754e+03
50%,0.425841,1.000000,0.000000,1.000000,0.000000,9.492940e+03,8.853187e+03
75%,0.612923,1.000000,0.000000,2.000000,0.000000,2.678135e+04,2.400200e+04
max,0.799999,5.000000,214.000000,1578.000000,0.047054,4.988745e+08,1.781794e+08


In [9]:

# Table 2 
df_banks_check = pd.read_parquet(OUT_BANK_STATE / "sim_bank_state_2016Q1.parquet")
print("Table 2")
print(f"  shape : {df_banks_check.shape}")
print(f"  cols  : {list(df_banks_check.columns)}")
display(df_banks_check.head(5))
display(df_banks_check[["equity_initial", "equity_final", "equity_loss",
                         "fail_round", "loss_frac_of_initial"]].describe())

Table 2
  shape : (1891409, 10)
  cols  : ['run_id', 'bank_id', 'equity_initial', 'equity_final', 'equity_delta', 'equity_loss', 'failed', 'fail_round', 'loss_frac_of_initial', 'is_initial_bank']


,run_id,bank_id,equity_initial,equity_final,equity_delta,equity_loss,failed,fail_round,loss_frac_of_initial,is_initial_bank
0,482993bf-4282-4b77-948a-80463413358c,3419,14554.0,10883.807311,-3670.192689,3670.192689,False,NaN,0.252178,True
1,9e5154fe-f8bd-48c4-ad67-4764c3f1517a,3301,11198.0,8107.475317,-3090.524683,3090.524683,False,NaN,0.275989,True
2,ef7136bc-6463-4c2f-b56e-b3f5399c30fa,2507,14223.0,8739.239266,-5483.760734,5483.760734,False,NaN,0.385556,True
3,168188f4-e980-43e9-8ab1-467921a47355,1165,40640.0,27255.443657,-13384.556343,13384.556343,False,NaN,0.329344,True
4,d7aefb2a-34db-486c-96ef-606cdd125de2,3072,18728.0,17493.347445,-1234.652555,1234.652555,False,NaN,0.065925,True


,equity_initial,equity_final,equity_loss,fail_round,loss_frac_of_initial
count,1.891409e+06,1.891409e+06,1.891409e+06,122156.000000,1.891409e+06
mean,1.289765e+07,1.270225e+07,1.954032e+05,1.261354,3.391485e-01
std,3.847429e+07,3.829993e+07,2.083324e+06,0.558684,1.176897e+00
min,0.000000e+00,-2.588872e+07,0.000000e+00,0.000000,0.000000e+00
25%,1.615800e+04,7.994399e+03,2.898000e+03,1.000000,4.229465e-03
50%,4.781800e+04,3.416900e+04,7.106000e+03,1.000000,1.456799e-01
75%,4.224927e+06,3.662830e+06,1.712800e+04,1.000000,4.822444e-01
max,2.081500e+08,2.081500e+08,1.781794e+08,4.000000,9.719668e+01


In [10]:
# For every run: Table 1 num_failed  ==  Table 2 rows where failed=True
t1 = df_runs_check[["run_id", "num_failed"]].copy()
t2_failed = (
    df_banks_check[df_banks_check["failed"]]
    .groupby("run_id")
    .size()
    .rename("t2_failed_count")
    .reset_index()
)
check = t1.merge(t2_failed, on="run_id", how="left")
check["t2_failed_count"] = check["t2_failed_count"].fillna(0).astype(int)

mismatches = check[check["num_failed"] != check["t2_failed_count"]]
print(f"Cross-table mismatches: {len(mismatches)}  (expect 0)")
assert len(mismatches) == 0, "Tables 1 and 2 are inconsistent!"

Cross-table mismatches: 0  (expect 0)
